# 🌍 Prototipo: Planeta rotando en proyección Orthographic — ERA5 real

Dataset: **ERA5 reanalysis-single-levels, 2m temperature** (`era5_t2m_2024-2026.nc`, 1464 pasos de tiempo, grilla global 0.25°).

Mejoras sobre `copernicus_planet_temp.ipynb`:

| Problema original | Solución en este prototipo |
|---|---|
| `imshow(..., transform=PlateCarree())` falla en Orthographic (`ValueError` de `warp_array`) | Se usa `pcolormesh`, que proyecta correctamente cualquier grilla regular |
| La "animación" vieja animaba niveles de altura, no rotaba el planeta | Animación A: rotación 360°; Animación B: evolución temporal (time-lapse) |
| `ax.clear()` + re-agregar features por frame (lentísimo) | Features y `mesh` se crean una vez; por frame solo `mesh.set_array()` |
| Carga completa del dataset → sin RAM (6 GB virtuales) | Lectura **lazy** con xarray: por frame se decodifica solo el slice de ~4 MB |
| Código duplicado por celda | Configuración centralizada + funciones reutilizables |
| Rango de color cambiaba entre frames | `vmin/vmax` fijos (-40 a 40 °C) → comparación consistente |

**Ambiente:** `.venv` (Python 3.12, creado con `uv`). Kernel: **Python (climate-3d)**.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Configuración centralizada ---
DATA_DIR   = Path(".")
ERA5_FILE  = DATA_DIR / "era5_t2m_2024-2026.nc"   # dataset descargado desde CDS
VAR_CANDIDATES = ("t2m", "stl2", "t")             # variables aceptadas
CENTER_LON, CENTER_LAT = -58.0, -23.0             # vista centrada en Paraguay
ASUNCION = {"nombre": "Asunci\u00f3n (PY)", "lat": -25.2637, "lon": -57.5759}
VMIN, VMAX = -40, 40                              # rango fijo en °C
CMAP       = "coolwarm"
N_FRAMES   = 60                                   # frames de la rotación (360°)
N_TIME_FRAMES = 72                                # frames del time-lapse
FPS        = 8                                    # más lento para apreciar el movimiento
STEP_DEG   = 360 / N_FRAMES

plt.rcParams["figure.dpi"] = 100
print("Entorno listo:")
import cartopy, matplotlib
print(f"  numpy {np.__version__} | matplotlib {matplotlib.__version__} | cartopy {cartopy.__version__}")

## 0. Descarga del dataset (solo la primera vez)

Usa tu API key de CDS configurada en `~/.cdsapirc`. Si `era5_t2m_2024-2026.nc` ya existe, la celda no hace nada. También puedes descargar por fuera con: `.venv\Scripts\python.exe descargar_era5.py`

In [ ]:
if ERA5_FILE.exists() and ERA5_FILE.stat().st_size > 0:
    print(f"Ya existe: {ERA5_FILE.name} ({ERA5_FILE.stat().st_size/1e9:.2f} GB) — no se descarga de nuevo")
else:
    import cdsapi
    client = cdsapi.Client()
    client.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": ["reanalysis"],
            "variable": ["2m_temperature"],
            "year": ["2024", "2025", "2026"],
            "month": ["01", "02", "03", "04", "05", "06",
                      "07", "08", "09", "10", "11", "12"],
            "day": ["01", "04", "07", "09", "12", "15",
                    "17", "20", "23", "25", "28", "31"],
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
    ).download(str(ERA5_FILE))
    print("Descarga completa:", ERA5_FILE)

## 1. Carga lazy del dataset

`load_field()` abre el NetCDF **sin leerlo completo** (lazy) y devuelve el DataArray 3D `(tiempo, lat, lon)` en Kelvin; la conversión a °C se hace por slice al leer cada frame (sin dask, la aritmética lazy sobre el array completo materializaría los 6 GB en RAM). Si no hubiera archivo, genera un campo sintético con ciclo diurno para que el prototipo siempre funcione.

In [ ]:
def synthetic_field(nlat=181, nlon=360, n_times=24):
    """Campo sintético global (°C) con ciclo diurno, para fallback sin archivo."""
    lat = np.linspace(90, -90, nlat)
    lon = np.linspace(0, 359, nlon)
    LAT, LON = np.meshgrid(lat, lon, indexing="ij")
    base = 30 * np.cos(np.deg2rad(LAT)) ** 2 - 17                    # gradiente ecuador-polos
    relief = (8 * np.sin(np.deg2rad(LON * 3)) * np.cos(np.deg2rad(LAT * 2))
              + 4 * np.sin(np.deg2rad(LON * 7 + 40)))
    t = np.arange(n_times)
    diurnal = 8 * np.sin(2 * np.pi * t[:, None, None] / n_times
                         - 2 * np.pi * LON[None] / 360)              # onda que gira con el sol
    data = base[None] + relief[None] + diurnal
    times = np.datetime64("2024-01-01T00:00") + t * np.timedelta64(1, "h")
    da = xr.DataArray(data, dims=("valid_time", "latitude", "longitude"),
                      coords={"valid_time": times, "latitude": lat, "longitude": lon})
    return da


def load_field(data_dir=DATA_DIR):
    """Devuelve (da_3D °C lazy, times). Usa el primer .nc/.grib válido; si no hay, sintético."""
    files = list(data_dir.glob("*.nc")) + list(data_dir.glob("*.grib")) + list(data_dir.glob("*.grib2"))
    for f in sorted(files, key=lambda p: p.stat().st_size, reverse=True):
        try:
            ds = xr.open_dataset(f)
        except Exception:
            continue
        for name in VAR_CANDIDATES:
            if name not in ds:
                continue
            da = ds[name]
            # colapsar dims que no sean tiempo/lat/lon (p. ej. expver, number)
            keep = ("valid_time", "time", "latitude", "longitude")
            da = da.isel({d: 0 for d in da.dims if d not in keep})
            tdim = "valid_time" if "valid_time" in da.dims else "time"
            if tdim not in da.dims:                      # archivo 2D -> eje de tiempo trivial
                da = da.expand_dims({tdim: [np.datetime64("NaT")]})
            # NOTA: se deja en Kelvin; restar 273.15 aqui leeria el archivo completo
            # (sin dask la aritmetica lazy no esta soportada). Se convierte por slice.
            print(f"Archivo real: {f.name} | variable '{name}' | pasos de tiempo: {da.sizes[tdim]}")
            return da, da[tdim].values
    print("Sin archivo real -> datos sintéticos")
    da = synthetic_field()
    return da, da["valid_time"].values


da, times = load_field()
n_times = len(times)
to_c = lambda arr: np.asarray(arr, dtype=np.float64) - 273.15   # K -> °C por slice

# Fecha por defecto para la animación de rotación (cámbiala a gusto)
T_TARGET = np.datetime64("2024-07-15T12:00")
T_IDX = int(np.argmin(np.abs(times - T_TARGET)))
print(f"Grilla {da.shape} | rango del primer paso: "
      f"{to_c(da[0]).min():.1f} a {to_c(da[0]).max():.1f} °C")
print(f"T_IDX = {T_IDX} -> {np.datetime_as_string(times[T_IDX], unit='h')} (UTC)")

## 2. Frame estático de control

Un vistazo al campo en `T_IDX` antes de animar.

In [ ]:
def draw_base_map(field, center_lon=0.0, fig=None, fig_size=(7, 7)):
    """GeoAxes Orthographic + mesh con 'field' (°C). Si se pasa 'fig', la limpia y reusa
    (permite recrear el axes con otro central_longitude -> rotación de cámara)."""
    if fig is None:
        fig = plt.figure(figsize=fig_size)
    else:
        fig.clf()
    ax = fig.add_subplot(111, projection=ccrs.Orthographic(central_longitude=center_lon,
                                                           central_latitude=CENTER_LAT))
    ax.set_global()
    mesh = ax.pcolormesh(da.longitude, da.latitude, field,
                         transform=ccrs.PlateCarree(),
                         cmap=CMAP, vmin=VMIN, vmax=VMAX, shading="auto")
    try:
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":", zorder=3)
    except Exception as exc:
        print(f"Features omitidas (¿sin internet?): {exc}")
    return fig, ax, mesh


fig, ax, mesh = draw_base_map(to_c(da[T_IDX].values), center_lon=CENTER_LON)
cbar = plt.colorbar(mesh, orientation="horizontal", pad=0.04, shrink=0.8, extend="both")
cbar.set_label("Temperatura 2 m (°C)")
cbar.set_ticks(np.arange(VMIN, VMAX + 1, 20))
ax.set_title(f"ERA5 t2m — {np.datetime_as_string(times[T_IDX], unit='h')} (UTC)")
plt.show()

## 3. Animación A — Planeta rotando 360° (cámara orbital)

Aquí **gira la proyección**, no los datos: en cada frame se recrea el GeoAxes con `central_longitude = frame × 6°`, de modo que el globo completo (continentes + campo de temperatura) rota como una vista orbital del planeta. Es más pesado por frame (re-render del mapa), pero es la rotación real. GIF con `PillowWriter` (no requiere ffmpeg).

In [ ]:
field0 = to_c(da[T_IDX].values)
fig = plt.figure(figsize=(7, 7))

def update_rot(frame):
    ang = frame * STEP_DEG
    # Rotación de CÁMARA: se recrea el GeoAxes con central_longitude=ang
    # (el globo entero —continentes y datos— gira frente al observador)
    _, ax, mesh = draw_base_map(field0, center_lon=ang, fig=fig)
    ax.set_title(f"ERA5 t2m {np.datetime_as_string(times[T_IDX], unit='h')} UTC — "
                 f"rotación {ang:.0f}°", fontsize=11)
    return mesh,

anim_rot = animation.FuncAnimation(fig, update_rot, frames=N_FRAMES,
                                   interval=1000 / FPS, blit=False)

gif_rot = DATA_DIR / "planeta_rotando.gif"
anim_rot.save(gif_rot, writer=animation.PillowWriter(fps=FPS))
plt.close(fig)
print("GIF guardado en:", gif_rot.resolve())

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(gif_rot)))

## 4. Animación B — Evolución temporal (time-lapse)

Vista fija sobre Paraguay; el campo avanza por `N_TIME_FRAMES` momentos elegidos uniformemente en los 2.7 años del dataset. Lectura lazy: cada frame decodifica solo su slice.

Incluye un **marcador con línea señalando Asunción, Paraguay**, y un recuadro con la **temperatura de ese instante** (píxel de grilla más cercano a la ciudad).

In [ ]:
idxs = np.linspace(0, n_times - 1, N_TIME_FRAMES).astype(int)

fig, ax, mesh = draw_base_map(to_c(da[T_IDX].values), center_lon=CENTER_LON)
title = ax.set_title("", fontsize=11)

# --- Marcador + línea + recuadro de temperatura para Asunción, Paraguay ---
ilat = int(np.abs(da.latitude.values - ASUNCION["lat"]).argmin())
ilon = int(np.abs(da.longitude.values - ASUNCION["lon"]).argmin())
ax.plot(ASUNCION["lon"], ASUNCION["lat"], "o", color="black", markersize=5,
        transform=ccrs.PlateCarree(), zorder=6)

pc_trans = ccrs.PlateCarree()._as_mpl_transform(ax)
label = ax.annotate(
    f"{ASUNCION['nombre']}\n{to_c(da[idxs[0]].values)[ilat, ilon]:.1f} °C",
    xy=(ASUNCION["lon"], ASUNCION["lat"]), xycoords=pc_trans,
    xytext=(ASUNCION["lon"] + 22, ASUNCION["lat"] - 17), textcoords=pc_trans,
    arrowprops=dict(arrowstyle="->", color="black", lw=1.3),
    fontsize=10, fontweight="bold", zorder=6,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black", alpha=0.85))

def update_time(frame):
    i = idxs[frame]
    field = to_c(da[i].values)
    mesh.set_array(field.ravel())
    label.set_text(f"{ASUNCION['nombre']}\n{field[ilat, ilon]:.1f} °C")
    title.set_text(f"ERA5 t2m — {np.datetime_as_string(times[i], unit='h')} (UTC)")
    return mesh, label, title

anim_time = animation.FuncAnimation(fig, update_time, frames=N_TIME_FRAMES,
                                    interval=1000 / FPS, blit=False)

gif_time = DATA_DIR / "planeta_evolucion.gif"
anim_time.save(gif_time, writer=animation.PillowWriter(fps=FPS))
plt.close(fig)
print("GIF guardado en:", gif_time.resolve())

In [ ]:
display(Image(filename=str(gif_time)))

## 5. Export opcional a MP4 (requiere ffmpeg en PATH) y siguientes pasos

In [ ]:
for name, anim in (("planeta_rotando.mp4", anim_rot), ("planeta_evolucion.mp4", anim_time)):
    try:
        anim.save(DATA_DIR / name, writer="ffmpeg", fps=FPS)
        print("MP4 guardado:", DATA_DIR / name)
    except Exception as exc:
        print(f"ffmpeg no disponible para {name}; se conserva el GIF.")
        break

### Notas y siguientes mejoras
- `expver`: los meses recientes de 2026 provienen de ERA5T (preliminar); CDS ya los entrega fusionados en la misma variable.
- Para otra fecha en la rotación: cambia `T_TARGET` en la celda 1 y re-ejecuta.
- Más frames: sube `N_FRAMES` / `N_TIME_FRAMES` (más peso en el GIF).
- Ideas: terminador día/noche con `cfeature.Nightshade`, contornos de isotermas con `ax.contour`, animar rotación + tiempo a la vez.